In [3]:
# -----------------------------------------------------------
# train_pipeline_optimized.py
# Author: Sahil
# Description: Optimized RandomForest pipeline (smaller + efficient)
# -----------------------------------------------------------

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# -----------------------------------------------------------
# 1️⃣ Load dataset
# -----------------------------------------------------------
df = pd.read_csv("modified_new_healthcare_dataset.csv")

# Clean column names
df.columns = df.columns.str.replace(' ', '_')

# Drop irrelevant columns
cols_to_drop = ["Name", "Doctor", "Hospital", "Room_Number", 
                "Date_of_Admission", "Discharge_Date"]
df = df.drop(columns=cols_to_drop)

# Target & features
target = "Billing_Amount"
X = df.drop(columns=[target])
y = df[target]

# -----------------------------------------------------------
# 2️⃣ Feature types
# -----------------------------------------------------------
categorical_features = [
    "Gender",
    "Blood_Type",
    "Medical_Condition",
    "Insurance_Provider",
    "Admission_Type",
    "Medication",
    "Test_Results"
]

numerical_features = ["Age", "Length_of_Stay"]

# -----------------------------------------------------------
# 3️⃣ Preprocessing (OPTIMIZED)
# -----------------------------------------------------------
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(
        handle_unknown="ignore",
        max_categories=20,       # 🔥 limits feature explosion
        sparse_output=False      # easier for tree models
    ), categorical_features),
    
    ("num", StandardScaler(), numerical_features)
])

# -----------------------------------------------------------
# 4️⃣ Optimized RandomForest
# -----------------------------------------------------------
model = RandomForestRegressor(
    n_estimators=80,          # 🔽 reduced trees
    max_depth=12,             # 🔽 limit depth
    min_samples_leaf=5,       # 🔽 smoother trees
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# -----------------------------------------------------------
# 5️⃣ Train/test split
# -----------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------------------------------------
# 6️⃣ Train
# -----------------------------------------------------------
pipeline.fit(X_train, y_train)
print("✅ Model training complete.")

# -----------------------------------------------------------
# 7️⃣ Evaluation
# -----------------------------------------------------------
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

print("\n📊 Model Performance Metrics:")
print(f"Train MAE:  {mean_absolute_error(y_train, y_train_pred):.2f}")
print(f"Test MAE:   {mean_absolute_error(y_test, y_test_pred):.2f}")
print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.2f}")
print(f"Test RMSE:  {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f}")
print(f"Train R²:   {r2_score(y_train, y_train_pred):.4f}")
print(f"Test R²:    {r2_score(y_test, y_test_pred):.4f}")

# -----------------------------------------------------------
# 8️⃣ Save with compression (VERY IMPORTANT)
# -----------------------------------------------------------
joblib.dump(pipeline, "billing_model.joblib", compress=3)

print("\n💾 Optimized model saved as billing_model.joblib")

✅ Model training complete.

📊 Model Performance Metrics:
Train MAE:  5425.10
Test MAE:   6078.92
Train RMSE: 8546.62
Test RMSE:  9733.09
Train R²:   0.8683
Test R²:    0.8311

💾 Optimized model saved as billing_model.joblib
